# Stocker — End-to-end Colab workflow

Self-contained: clones the repo, installs deps, builds the dataset, loads
**`google/gemma-4-E4B-it`** in-process via `transformers` (4-bit BnB),
pre-caches all 7 specialist votes, runs **GRPO** on the moderator LoRA via
TRL, and saves loss / reward plots + the trained adapter.

Designed for a free **Colab T4** (16 GB VRAM) — no separate vLLM server
needed. If you have an L4/A100, drop `load_in_4bit=False` for full bf16.

Outputs live under `training/runs/<timestamp>/`:
- `moderator-lora/` — trained PEFT adapter
- `loss.png`, `reward.png` — training curves
- `eval_pre.json`, `eval_post.json` — pre/post backtest reports

> Tip: `Runtime → Change runtime type → T4 GPU` before running.

In [ ]:
# Install — keep Colab's pre-baked pandas/numpy (TF + cudf depend on them);
# only upgrade what we strictly need.
#
# Big upgrades (transformers/trl/peft) are intentional — Colab ships old.
# Everything else uses --upgrade-strategy only-if-needed so we don't
# break google-colab/tensorflow/gradio/etc.
!pip install -q -U 'transformers>=4.55' 'trl>=0.11' 'peft>=0.13' 'accelerate>=1.0' 'bitsandbytes>=0.43' 'datasets>=3.0'
!pip install -q --upgrade-strategy=only-if-needed yfinance mplfinance pyarrow 'pydantic>=2' pydantic-settings 'openai>=1' tensorboard 'huggingface_hub>=1.10'
# Indicators are computed in-repo (app/data/indicators.py) — no pandas-ta.

In [ ]:
import os, sys, pathlib

# 1) Already in a stocker repo? (running from VSCode on a local clone, or
#    the repo was manually uploaded to Colab). Detect and skip the clone.
def _is_stocker_root(p: str) -> bool:
    return os.path.isfile(os.path.join(p, "app", "council", "specialists.py"))

CANDIDATES = [os.getcwd(), "/content/stocker", "/workspace/stocker"]
WORKDIR = next((c for c in CANDIDATES if _is_stocker_root(c)), None)

if WORKDIR is None:
    # 2) Not present — clone. EDIT THIS URL before running on a fresh Colab.
    REPO_URL = "https://github.com/<your-username>/stocker.git"
    WORKDIR  = "/content/stocker"
    assert "<your-username>" not in REPO_URL, (
        "Edit REPO_URL in this cell to your fork before running on a fresh runtime."
    )
    !git clone {REPO_URL} {WORKDIR}
    assert _is_stocker_root(WORKDIR), f"Clone failed — {WORKDIR}/app/council/ missing."

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print("Working dir:", WORKDIR)

In [ ]:
# Gemma 4 is Apache-2.0 (not gated) — token just lifts download rate limits.
from huggingface_hub import login
import getpass
login(token=getpass.getpass("HF token (read scope is enough): "))

In [ ]:
# Build the bundled dataset (yfinance + indicators + chart PNGs).
# Idempotent — skip if already done.
!python scripts/build_dataset.py
!python scripts/validate_tasks.py

In [ ]:
from app.council.llm import TransformersLLMClient

# 4-bit BnB by default; drop load_in_4bit=False on L4/A100.
client = TransformersLLMClient.from_pretrained(
    "google/gemma-4-E4B-it",
    load_in_4bit=True,
)
print("Model loaded on:", client.model.device)

In [ ]:
# Pre-cache the 7 specialists' votes for every (task, step). Specialists
# are FROZEN, so this runs once and is reused across every GRPO step.
from app.council.runner import Council
from app.core.environment import StockerEnv
from app.core.tasks import list_task_ids

council = Council(client=client, use_cache=True)
total = 0
for task_id in list_task_ids():
    env = StockerEnv(task_id=task_id)
    obs = env.reset().observation
    while True:
        for sp in council.specialists:
            council._cached_vote(sp, obs)   # writes .cache/council/<role>/...
            total += 1
        result = env.step({"side": "hold", "quantity": 0})
        if result.done:
            break
        obs = result.observation
print(f"cached {total} specialist votes across {len(list_task_ids())} tasks")

In [ ]:
# Pre-training baseline rollout (specialists from cache, moderator = base Gemma)
!python -m training.eval_rollout --out training/runs/eval_pre
!cat training/runs/eval_pre/summary.csv

In [ ]:
# GRPO on the moderator LoRA. Specialist votes are read from cache —
# only the moderator is re-rolled per training step, so this is fast.
#
# Knobs: --num-generations is K candidates per prompt (GRPO group size).
# On a free T4 keep batch_size=1, grad_accum=8.
!python -m training.train_grpo \
    --epochs 2 \
    --num-generations 8 \
    --batch-size 1 \
    --grad-accum 8 \
    --lora-rank 16 \
    --lr 5e-6

In [ ]:
# Post-training eval — load the trained adapter into the same client.
import glob, os
RUN_DIR = sorted(glob.glob("training/runs/grpo_*"))[-1]
LORA_DIR = os.path.join(RUN_DIR, "moderator-lora")
print("Using LoRA:", LORA_DIR)

# Attach adapter and re-run eval. The adapter name "moderator" matches what
# Moderator.decide() requests via extra_body={'lora_request': {'name': ...}}
from peft import PeftModel
client.model = PeftModel.from_pretrained(client.model, LORA_DIR, adapter_name="moderator")
client.moderator_lora = "moderator"

!python -m training.eval_rollout --moderator-lora moderator --out training/runs/eval_post
!cat training/runs/eval_post/summary.csv

In [ ]:
# Show training + eval curves inline
from IPython.display import Image, display
import os
for p in [f"{RUN_DIR}/loss.png", f"{RUN_DIR}/reward.png",
          "training/runs/eval_pre/reward_curve.png",
          "training/runs/eval_post/reward_curve.png",
          "training/runs/eval_pre/portfolio_curve.png",
          "training/runs/eval_post/portfolio_curve.png"]:
    if os.path.exists(p):
        print(p)
        display(Image(p))